# Lesson 6: The GPT-2 decoder block

Lesson 5 computed a causal attention result. GPT-2 wraps that operation in a decoder block with two pre-normalized residual updates:

\[x \leftarrow x + \operatorname{Attention}(\operatorname{LayerNorm}_1(x))\]
\[x \leftarrow x + \operatorname{MLP}(\operatorname{LayerNorm}_2(x))\]

The attention sublayer mixes information across token positions. The MLP then transforms each position independently. The residual paths preserve each sublayer's incoming representation and add its update.

We reuse Lesson 4's three-token, two-channel input. A real GPT-2 block has width 768, 12 attention heads, and an MLP width of 3,072; our tiny dimensions and fixed weights keep the steps inspectable. The toy Q/K/V and attention-output projections are identity matrices, while the MLP weights are small fixed values shared with C and Java. Nothing is trained in this lesson.

**Read the companions:** [C block_lesson.c](../../c/src/block_lesson.c) and [Java BlockLesson.java](../../java/src/main/java/learning/karpathy/gpt2/BlockLesson.java). Run them with -Action lesson -Lesson 6.


In [ ]:
import math
import sys

import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")


## 1. LayerNorm, causal attention, and the first residual path

LayerNorm normalizes the channels independently for each token. PyTorch applies learned scale and shift parameters after normalization; the toy starts with scale 1 and shift 0. The C and Java implementations show the mean, variance, epsilon, and per-channel calculation directly.

After normalization, we reuse Lesson 5's single-head causal attention. The Q/K/V and attention-output projections are identity matrices here, so this section focuses on where normalization and the residual addition sit in the block.


In [ ]:
SEQUENCE_LENGTH = 3
EMBEDDING_SIZE = 2
MLP_WIDTH = 4 * EMBEDDING_SIZE
EPSILON = 1e-5
dtype = torch.float64

X = torch.tensor(
    [[1.1, 0.0], [0.0, 1.2], [1.3, 1.3]],
    dtype=dtype,
)
gamma_1 = torch.ones(EMBEDDING_SIZE, dtype=dtype)
beta_1 = torch.zeros(EMBEDDING_SIZE, dtype=dtype)
W_Q = torch.eye(EMBEDDING_SIZE, dtype=dtype)
W_K = torch.eye(EMBEDDING_SIZE, dtype=dtype)
W_V = torch.eye(EMBEDDING_SIZE, dtype=dtype)
W_ATTENTION_OUT = torch.eye(EMBEDDING_SIZE, dtype=dtype)

x_norm_1 = F.layer_norm(X, (EMBEDDING_SIZE,), gamma_1, beta_1, EPSILON)
Q, K, V = F.linear(x_norm_1, W_Q), F.linear(x_norm_1, W_K), F.linear(x_norm_1, W_V)
attention_context = F.scaled_dot_product_attention(
    Q[None, None, :, :],
    K[None, None, :, :],
    V[None, None, :, :],
    is_causal=True,
)[0, 0]
attention_branch = F.linear(attention_context, W_ATTENTION_OUT)
after_attention = X + attention_branch

print("Input residual stream X (T, C):\n", X)
print("LayerNorm 1 output:\n", x_norm_1)
print("Causal attention context:\n", attention_context)
print("Attention branch output:\n", attention_branch)
print("After the first residual, x + attention(LayerNorm 1(x)):\n", after_attention)


## 2. The position-wise MLP and the second residual path

The MLP first expands each token vector from C channels to 4C, applies GELU, then projects back to C. It uses the same weights for every token, but unlike attention it does not mix information between positions.

This is the tanh-approximate GELU used by the reference. The C and Java versions spell out its formula; PyTorch provides it as a library operation. As in GPT-2, LayerNorm 2 is applied to the result of the first residual update before the MLP runs.


In [ ]:
gamma_2 = torch.ones(EMBEDDING_SIZE, dtype=dtype)
beta_2 = torch.zeros(EMBEDDING_SIZE, dtype=dtype)
x_norm_2 = F.layer_norm(after_attention, (EMBEDDING_SIZE,), gamma_2, beta_2, EPSILON)

# F.linear stores weights as (output channels, input channels).
W_MLP_IN = torch.tensor(
    [[0.20, -0.10], [-0.10, 0.25], [0.30, 0.05], [0.00, 0.20],
     [-0.20, 0.10], [0.10, -0.15], [0.15, 0.00], [-0.05, 0.30]],
    dtype=dtype,
)
b_MLP_IN = torch.zeros(MLP_WIDTH, dtype=dtype)
W_MLP_OUT = torch.tensor(
    [[0.10, -0.10, 0.20, 0.00, 0.10, -0.15, 0.05, 0.10],
     [-0.20, 0.15, 0.05, 0.20, 0.10, 0.05, -0.05, 0.20]],
    dtype=dtype,
)
b_MLP_OUT = torch.zeros(EMBEDDING_SIZE, dtype=dtype)

mlp_pre_activation = F.linear(x_norm_2, W_MLP_IN, b_MLP_IN)
mlp_activated = F.gelu(mlp_pre_activation, approximate="tanh")
mlp_branch = F.linear(mlp_activated, W_MLP_OUT, b_MLP_OUT)
block_output = after_attention + mlp_branch

print("LayerNorm 2 output:\n", x_norm_2)
print("MLP linear output before GELU (T, 4C):\n", mlp_pre_activation)
print("MLP after GELU:\n", mlp_activated)
print("MLP branch output:\n", mlp_branch)
print("Block output after the second residual:\n", block_output)


## 3. Verify the block equations

The reference code adds each sublayer's output to its input. These checks make both residual equations explicit and verify that the library GELU matches the formula used by the C and Java companions.


In [ ]:
gelu_manual = 0.5 * mlp_pre_activation * (
    1.0 + torch.tanh(
        math.sqrt(2.0 / math.pi)
        * (mlp_pre_activation + 0.044715 * mlp_pre_activation.pow(3))
    )
)
torch.testing.assert_close(mlp_activated, gelu_manual, rtol=1e-12, atol=1e-12)
torch.testing.assert_close(after_attention, X + attention_branch, rtol=0.0, atol=0.0)
torch.testing.assert_close(block_output, after_attention + mlp_branch, rtol=0.0, atol=0.0)
assert torch.isfinite(block_output).all()
print("GELU library operation matches the explicit tanh approximation: yes")
print("Both residual equations hold and all block outputs are finite: yes")
print(f"Block input/output shape: {tuple(X.shape)}; MLP hidden shape: {tuple(mlp_activated.shape)}")


## Connect this to GPT-2

The reference Block.forward follows exactly these two updates. Its CausalSelfAttention contains the learned Q/K/V and output projections; its MLP expands to four times the embedding width, applies tanh-approximate GELU, and projects back. GPT-2 stacks 12 blocks, then applies a final LayerNorm and a vocabulary projection to produce next-token logits.

The tutorial model uses trained parameters. Our identity attention matrices and fixed MLP values demonstrate data flow and shapes only. The next step can assemble token embeddings, a block, final normalization, and logits into a tiny end-to-end GPT forward pass.

**Try it:** set one of the MLP weights to zero and compare the block output. Then temporarily remove one residual addition and observe that the corresponding input representation is no longer carried forward.

**References:** Karpathy's [MLP and Block source](../../build-nanogpt/train_gpt2.py#L42) and [PyTorch LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html), [GELU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html).
